<a href="https://colab.research.google.com/github/AdelineKwakye/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
# TODO
removed = df.duplicated().sum() # count number of duplicated rows that will be removed
clean = df.drop_duplicates().copy()

log('duplicates', 'dropped duplicate rows', removed)

[duplicates] dropped duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
# TODO
clean['price'] = (clean['price']
                  .str.replace('$','',regex=False)
                  .str.strip()
                  .astype(float)) # strip $, whitespace, and convert to float
log('price', 'stripped the $ from beginning and whitespaces and converted to float', len(clean))

[price] stripped the $ from beginning and whitespaces and converted to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# TODO
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce') # convert to numeric

missing = clean['qty'].isna().sum() # count missing qty
negative = (clean['qty'] < 0 ).sum() # count negative


# clean['revenue_before'] = clean['qty'] * clean['price']
# print("Revenue with missing/negative rows kept:",
      # round(clean['revenue_before'].sum(), 2))
# revenue with missing/negative rows = 4594.5
clean = clean[clean['qty'].notna() & (clean['qty'] > 0 )].copy() # keep the not na values and the positive values
log('qty', 'converted qty to numeric', len(clean))
log('missing values', 'dropped the missing values', missing)
log('negative values', 'dropped the negative values', negative)

[qty] converted qty to numeric (275 row(s))
[missing values] dropped the missing values (12 row(s))
[negative values] dropped the negative values (13 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(clean['item'].value_counts())
before = clean['item'].nunique() # count how many items prior to canonicalizing
ITEM_MAP = {
    'foam finger': 'Foam Finger',
    'cheese burger': 'Cheeseburger',
    'rain poncho': 'Rain Poncho'

}  # canonicalize item
clean['item'] = clean['item'].replace(ITEM_MAP)
print(clean['item'].value_counts())
after = clean['item'].nunique() # count how many items after canonicalizing
log('item', 'built a mapping dict', before - after)

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64
[item] built a mapping dict (3 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
# TODO
print(clean['category'].value_counts())
before = clean['category'].nunique() # count how many category items prior to canonicalizing

CATEGORY_MAP = {
    'food': 'Food',
    'Merch': 'Merch',
    'rain-gear': 'RainGear',
    'Apparel': 'Apparel'
} # canonicalize category

clean['category'] = clean['category'].replace(CATEGORY_MAP)
print(clean['category'].value_counts())
after = clean['category'].nunique() # count how many category items after canonicalizing
log('category', 'built a mapping dict and kept Merch and Apparel separate as there might be something that distinguishes them', before - after)


category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
category
Food        95
RainGear    86
Merch       51
Apparel     43
Name: count, dtype: int64
[category] built a mapping dict and kept Merch and Apparel separate as there might be something that distinguishes them (2 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert clean.duplicated().sum() == 0
assert clean['qty'].min() >= 1
assert clean['price'].dtype == float
# TODO: assert something about item
assert clean['item'].nunique() == 3
# TODO: assert something about category
assert clean['category'].nunique() == 4
print('clean:', clean.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
# TODO
clean['revenue'] = clean['qty'] * clean['price']
revenue_by_category = clean.groupby('category')['revenue'].sum().sort_values(ascending=False)

print(revenue_by_category.round(2))
print('Overall total:', round(clean['revenue'].sum(), 2))

category
Food        1656.0
RainGear    1512.0
Merch        856.5
Apparel      715.5
Name: revenue, dtype: float64
Overall total: 4740.0


**What I would tell the vendor:** to stock up more on Food because that's the category that brings in the most revenue of $1656.

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped duplicate rows,15
1,price,stripped the $ from beginning and whitespaces ...,300
2,qty,converted qty to numeric,275
3,missing values,dropped the missing values,12
4,negative values,dropped the negative values,13
5,item,built a mapping dict,3
6,category,built a mapping dict and kept Merch and Appare...,2


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a) The cleaning step that changed my revenue total the most was the decision to drop the missing/negative rows. With the missing/negative rows not included we get a revenue of 4740 dollars. However with those rows included we get a revenue of 4594.5 dollars. Making it a difference of 145.50 dollars.

b) The decision to keep Merch and Apparel separate, another option would've been to combine the two because in theory Merch and Apparel are essentially the same thing. However I don't think that combining these items as one would change the revenue it would just change how the revenue is grouped. I decided to keep it separate just incase there was a logical distinction between the two.